# Práctico 4 · Kafka en el clúster del curso

**CDD7018 · Taller Big Data · Grupo 4**  
**Dataset:** Amazon Reviews 2023 · Software

Este notebook presenta únicamente la evidencia final del práctico, organizada según las partes solicitadas por el profesor. Se eliminaron las pruebas exploratorias que no forman parte del resultado final.

**Archivos de apoyo en la carpeta:** `g4.py`, `panel.py`, `productor.py` y `consumidor.py`.

## Configuración inicial

In [1]:
%run panel.py

TOPIC = "g4.resenas"
GRUPO = "g4.top"

print("Topic:", TOPIC)
print("Grupo:", GRUPO)
print("Brokers:", B)

Topic: g4.resenas
Grupo: g4.top
Brokers: kafka1:9092,kafka2:9092,kafka3:9092


## 1. Diseño del stream

**Evento:** cada evento representa una reseña de Amazon Software. El JSON contiene `timestamp`, `parent_asin`, `user_id`, `rating`, `helpful_vote`, `verified_purchase` y `title`. Se omite `text` porque es el campo de mayor tamaño y no es necesario para las métricas del práctico.

**Tiempo de evento:** se utiliza `timestamp`, que representa cuándo ocurrió la reseña. En el archivo original está en milisegundos y `g4.py` lo transforma al formato ISO `AAAA-MM-DDTHH:MM:SS`.

**Clave:** se utiliza `parent_asin`, identificador del producto. Así, todas las reseñas del mismo producto llegan a la misma partición, mantienen su orden relativo y pueden agregarse correctamente por producto.

**Particiones:** el topic se configura con **6 particiones**, **factor de replicación 3** y **retención de 14 días**. Las seis particiones permiten hasta seis consumidores activos en paralelo dentro de un mismo *consumer group*.

In [2]:
from kafka.admin import NewTopic

if TOPIC not in admin.list_topics():

    admin.create_topics([
        NewTopic(
            TOPIC,
            num_partitions=6,
            replication_factor=3,
            topic_configs={
                "retention.ms": str(
                    14 * 24 * 3600 * 1000
                )
            }
        )
    ])

    admin.wait_for_topics([TOPIC])

    print("Topic creado correctamente:", TOPIC)

else:
    print("El topic ya existe:", TOPIC)

Topic creado correctamente: g4.resenas


In [3]:
estado(TOPIC)

partición 0 · líder 1 · réplicas [1, 2, 3] · ISR [1, 2, 3]
partición 1 · líder 2 · réplicas [2, 3, 1] · ISR [2, 3, 1]
partición 2 · líder 3 · réplicas [3, 1, 2] · ISR [3, 1, 2]
partición 3 · líder 2 · réplicas [2, 3, 1] · ISR [2, 3, 1]
partición 4 · líder 3 · réplicas [3, 1, 2] · ISR [3, 1, 2]
partición 5 · líder 1 · réplicas [1, 2, 3] · ISR [1, 2, 3]


### Interpretación del estado del topic

El topic `g4.resenas` quedó creado con 6 particiones y factor de replicación 3. Los líderes están distribuidos entre los tres brokers del clúster y las tres réplicas de cada partición aparecen en **ISR (In-Sync Replicas)**, por lo que estaban sincronizadas con su líder al momento de la medición.

## 2. Productor

La carga se realizó en dos etapas.

### Carga rápida

```bash
python productor.py --tasa 0 --max 100000
```

Resultado real:

```text
100,000 eventos para g4.resenas, desde 2022-01-01T00:01:02
listo: 100,000 eventos en 5.4 s (18,361 ev/s)
```

Después de esa carga se verificó la distribución de offsets por partición:

In [5]:
fin(TOPIC)

{0: 15406, 1: 13644, 2: 20831, 3: 19178, 4: 17254, 5: 13687}

### Distribución de eventos

| Partición | Eventos |
|---:|---:|
| 0 | 15.406 |
| 1 | 13.644 |
| 2 | 20.831 |
| 3 | 19.178 |
| 4 | 17.254 |
| 5 | 13.687 |
| **Total** | **100.000** |

El promedio teórico sería aproximadamente 16.667 eventos por partición. El reparto no es perfectamente uniforme porque la clave es `parent_asin`: productos con muchas reseñas concentran más eventos en la partición determinada por su hash. Este desbalance es coherente con la decisión de mantener todas las reseñas de un producto juntas.

In [6]:
mirar(TOPIC, 0, 0, 3)

0 B08V8Z5KY8 {"timestamp": "2022-01-01T00:16:15", "parent_asin": "B08V8Z5KY8", "user_id": "AEWHGUBQUC4W7V2M3OGC6A232RBA", "rating": 4.0, "helpful_vote": 0, "verified_purchase": true, "title": "I have had this for a year now and love it!"} · llegó: 2026-10-07 20:14:10 UTC
1 B01EAY1XPW {"timestamp": "2022-01-01T00:18:57", "parent_asin": "B01EAY1XPW", "user_id": "AFHENRRJBGQ3CV372MBKUJTCG2VQ", "rating": 5.0, "helpful_vote": 0, "verified_purchase": true, "title": "Good free app"} · llegó: 2026-10-07 20:14:10 UTC
2 B016LKXSXS {"timestamp": "2022-01-01T00:58:09", "parent_asin": "B016LKXSXS", "user_id": "AFUOK4LTJ2U5ASTDIDG5QX3AAP7A", "rating": 5.0, "helpful_vote": 1, "verified_purchase": true, "title": "Great app"} · llegó: 2026-10-07 20:14:10 UTC


### Tiempo de evento vs. tiempo de llegada

La función `mirar()` muestra dos tiempos distintos:

- **Tiempo de evento:** el `timestamp` almacenado en el JSON, que representa cuándo ocurrió realmente la reseña.
- **Tiempo de llegada:** el instante en que Kafka recibió el mensaje durante esta reproducción histórica.

Para métricas como “reseñas por hora” se utilizaría el **tiempo de evento**, porque representa el momento real del fenómeno analizado.

### Productor en vivo

Después de los primeros 100.000 eventos se continuó sin repetir registros:

```bash
python productor.py --tasa 200 --desde 100000
```

Resultado real:

```text
listo: 81,675 eventos en 408.4 s (200 ev/s)
```

Por lo tanto se reprodujeron las **181.675 reseñas disponibles desde 2022**:

**100.000 + 81.675 = 181.675 eventos**.

## 3. Consumidor: métrica en vivo y alerta

El consumidor mantiene por producto la cantidad de reseñas, la suma de ratings y el **rating promedio**. La tabla auxiliar `meta` permite mostrar también el nombre del producto y la tienda.

La alerta del Grupo 4 se activa cuando un producto alcanza **10 reseñas de 1 estrella durante un mismo día de tiempo de evento**.

Durante la ejecución se observaron, entre otras, las siguientes alertas reales:

- **Paramount+**, tienda **CBS Interactive**, día **2022-02-06**.
- **Stremium - Free Live TV platform & Cloud DVR!**, tienda **Stremium Inc.**, día **2022-02-09**.

El consumidor pudo seguir una producción de **200 eventos/s** y en condiciones estables llegó a mostrar **lag 0**.

## 4. Consumer Groups y rebalanceo

Se ejecutaron dos consumidores simultáneamente dentro del grupo `g4.top`.

In [9]:
miembros("g4.top")

estado del grupo: Stable
  consumidor2 · particiones [0, 1, 2]
   sebastian · particiones [3, 4, 5]


In [10]:
lag("g4.top", TOPIC)

partición 0 · commit   27,257 · fin   27,320 · lag       63
partición 1 · commit   25,098 · fin   25,171 · lag       73
partición 2 · commit   35,597 · fin   35,661 · lag       64
partición 3 · commit   31,973 · fin   32,018 · lag       45
partición 4 · commit   30,006 · fin   30,077 · lag       71
partición 5 · commit   26,690 · fin   26,754 · lag       64
lag total del grupo g4.top: 380


Kafka distribuyó las seis particiones de forma equilibrada entre los dos consumidores. En ese instante el grupo presentaba un lag total de **380 eventos**.

In [11]:
miembros("g4.top")

estado del grupo: Stable
   sebastian · particiones [0, 1, 2, 3, 4, 5]


In [12]:
lag("g4.top", TOPIC)

partición 0 · commit   28,074 · fin   28,074 · lag        0
partición 1 · commit   25,917 · fin   25,917 · lag        0
partición 2 · commit   36,449 · fin   36,449 · lag        0
partición 3 · commit   32,780 · fin   32,780 · lag        0
partición 4 · commit   30,992 · fin   30,992 · lag        0
partición 5 · commit   27,463 · fin   27,463 · lag        0
lag total del grupo g4.top: 0


### Interpretación del rebalanceo

Al cerrar `consumidor2`, Kafka realizó un rebalanceo y `sebastian` recibió las seis particiones `[0,1,2,3,4,5]`. Posteriormente el grupo alcanzó nuevamente al productor y el lag volvió a **0**.

Kafka redistribuye las **particiones y offsets**, pero no transfiere el estado que cada proceso Python mantiene en memoria. Por eso los contadores, sumas y alertas acumulados por un consumidor no pasan automáticamente a otro durante un rebalanceo.

En este práctico la métrica principal está alineada con la clave `parent_asin`, lo que ayuda a mantener cada producto dentro de una sola partición. Una agregación global por un atributo distinto de la clave, por ejemplo `store`, requeriría una nueva redistribución o un motor de streaming con estado distribuido.

### Segundo consumer group

Se creó un grupo independiente denominado `g4.auditoria`.

In [13]:
miembros("g4.auditoria")

estado del grupo: Stable
   auditoria · particiones [0, 1, 2, 3, 4, 5]


In [14]:
lag("g4.auditoria", TOPIC)

partición 0 · commit   28,074 · fin   28,074 · lag        0
partición 1 · commit   25,917 · fin   25,917 · lag        0
partición 2 · commit   36,449 · fin   36,449 · lag        0
partición 3 · commit   32,780 · fin   32,780 · lag        0
partición 4 · commit   30,992 · fin   30,992 · lag        0
partición 5 · commit   27,463 · fin   27,463 · lag        0
lag total del grupo g4.auditoria: 0


`g4.auditoria` recibió las seis particiones y alcanzó **lag 0** sin afectar a `g4.top`. Esto demuestra que cada *consumer group* mantiene sus propios offsets y puede leer el mismo topic de manera independiente.

## 5. Experimento de lag

Para medir el crecimiento del lag sin alterar nuevamente el topic principal `g4.resenas`, se utilizó un topic auxiliar `g4.lagtest` con seis particiones y replicación 3.

El experimento mantuvo un productor a **200 eventos/s** durante **30 segundos** mientras el grupo `g4.lag` permanecía sin consumidores activos.

In [42]:
from pathlib import Path
from kafka import TopicPartition
from kafka.admin import NewTopic
import subprocess
import time

LAG_TOPIC = "g4.lagtest"

# Crear el topic auxiliar si no existe
if LAG_TOPIC not in admin.list_topics():
    admin.create_topics([
        NewTopic(
            LAG_TOPIC,
            num_partitions=6,
            replication_factor=3,
            topic_configs={"retention.ms": str(14 * 24 * 3600 * 1000)}
        )
    ])
    admin.wait_for_topics([LAG_TOPIC])

# Crear temporalmente una versión del productor que publique en g4.lagtest
g4_txt = Path("g4.py").read_text()
Path("g4_lag.py").write_text(
    g4_txt.replace('"g4.resenas"', '"g4.lagtest"', 1)
)

prod_txt = Path("productor.py").read_text()
Path("productor_lagtest.py").write_text(
    prod_txt.replace(
        "from g4 import TOPIC, CLAVE, TIEMPO, cargar",
        "from g4_lag import TOPIC, CLAVE, TIEMPO, cargar"
    )
)

def lag_total_valor(grupo, topic):
    commits = admin.list_group_offsets([grupo]).get(grupo, {})
    total = 0
    for particion, ultimo in fin(topic).items():
        info = commits.get(TopicPartition(topic, particion))
        commit = info.offset if info is not None and info.offset >= 0 else 0
        total += ultimo - commit
    return total

eventos_actuales = sum(fin(LAG_TOPIC).values())
print("Eventos actuales en g4.lagtest:", eventos_actuales)
print("Se continuará desde:", eventos_actuales)

productor = subprocess.Popen(
    [
        "python", "productor_lagtest.py",
        "--tasa", "200",
        "--desde", str(eventos_actuales),
        "--max", "30000"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

print("Productor lanzado. Esperando que realmente empiece a publicar...")

inicio_topic = sum(fin(LAG_TOPIC).values())

for i in range(60):
    time.sleep(1)
    ahora = sum(fin(LAG_TOPIC).values())
    if ahora > inicio_topic + 100:
        print("Productor confirmado en ejecución.")
        print("Eventos nuevos detectados:", ahora - inicio_topic)
        break
else:
    raise RuntimeError("No se detectó producción de nuevos eventos.")

miembros("g4.lag")

lag_inicial = lag_total_valor("g4.lag", LAG_TOPIC)
print("\nLag inicial:", lag_inicial)
print("Midiendo 30 segundos SIN consumidor...")

t0 = time.time()
time.sleep(30)

lag_final = lag_total_valor("g4.lag", LAG_TOPIC)
transcurrido = time.time() - t0

print(f"\nTiempo transcurrido: {transcurrido:.1f} s")
print("Lag final:", lag_final)
print("Crecimiento del lag:", lag_final - lag_inicial)
print("Esperado aproximadamente:", 200 * 30)
print("¿Productor sigue activo?:", productor.poll() is None)

# Limpieza del proceso/archivos auxiliares
if productor.poll() is None:
    productor.terminate()
Path("g4_lag.py").unlink(missing_ok=True)
Path("productor_lagtest.py").unlink(missing_ok=True)

Eventos actuales en g4.lagtest: 126000
Se continuará desde: 126000
Productor lanzado. Esperando que realmente empiece a publicar...
Productor confirmado en ejecución.
Eventos nuevos detectados: 179
estado del grupo: Empty

Lag inicial: 39779
Midiendo 30 segundos SIN consumidor...

Tiempo transcurrido: 30.0 s
Lag final: 45781
Crecimiento del lag: 6002
Esperado aproximadamente: 6000
¿Productor sigue activo?: True


### Resultado

El lag aumentó de **39.779** a **45.781** eventos durante 30 segundos.

**Crecimiento observado: 45.781 − 39.779 = 6.002 eventos**

El valor teórico esperado era:

**200 eventos/s × 30 s = 6.000 eventos**

La diferencia fue de solo **2 eventos**, por lo que la medición coincide prácticamente de forma exacta con `tasa × segundos`.

Después de reactivar el consumidor, éste recibió nuevamente las seis particiones y procesó el atraso hasta alcanzar **lag 0**, demostrando la recuperación del *consumer group* a partir de sus offsets confirmados.

## 6. Commit manual y garantía at-least-once

Para demostrar la semántica *at-least-once* de forma controlada se utilizó el grupo nuevo `g4.manual2`, `enable_auto_commit=False` y una sola partición del topic.

Primero se procesan 1.000 eventos y se cierra el consumidor **sin commit**, simulando una caída.

In [48]:
from kafka import KafkaConsumer, TopicPartition

MANUAL_GROUP = "g4.manual2"
tp = TopicPartition(TOPIC, 0)

def leer_particion(c, tp, n):
    vistos = []

    while len(vistos) < n:
        lotes = c.poll(
            timeout_ms=1000,
            max_records=n - len(vistos)
        )

        for _, mensajes in lotes.items():
            for m in mensajes:
                vistos.append(
                    (m.partition, m.offset)
                )

    return vistos


c1 = KafkaConsumer(
    bootstrap_servers=B,
    group_id=MANUAL_GROUP,
    enable_auto_commit=False
)

c1.assign([tp])

# Grupo nuevo: comenzar desde el principio
offset_commit_antes = c1.committed(tp)

if offset_commit_antes is None:
    c1.seek_to_beginning(tp)
else:
    c1.seek(tp, offset_commit_antes)

vistos_1 = leer_particion(c1, tp, 1000)

print("Commit antes de leer:", offset_commit_antes)
print("Primera lectura:", len(vistos_1), "eventos")
print("Primeros offsets:", vistos_1[:10])
print("Último offset:", vistos_1[-1])

# Simulamos caída: cerrar SIN commit
c1.close()

Commit antes de leer: None
Primera lectura: 1000 eventos
Primeros offsets: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9)]
Último offset: (0, 999)


In [49]:
c2 = KafkaConsumer(
    bootstrap_servers=B,
    group_id=MANUAL_GROUP,
    enable_auto_commit=False
)

c2.assign([tp])

offset_commit_despues_caida = c2.committed(tp)

print(
    "Commit después de la caída:",
    offset_commit_despues_caida
)

if offset_commit_despues_caida is None:
    c2.seek_to_beginning(tp)
else:
    c2.seek(tp, offset_commit_despues_caida)

vistos_2 = leer_particion(c2, tp, 1000)

repetidos = len(
    set(vistos_1) &
    set(vistos_2)
)

print("Segunda lectura:", len(vistos_2), "eventos")
print("Primeros offsets:", vistos_2[:10])
print("Eventos procesados dos veces:", repetidos)

Commit después de la caída: None
Segunda lectura: 1000 eventos
Primeros offsets: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9)]
Eventos procesados dos veces: 1000


In [50]:
c2.commit()

print(
    "Offset confirmado después del commit:",
    c2.committed(tp)
)

c2.close()

Offset confirmado después del commit: 1000


### Resultado del experimento at-least-once

La primera lectura procesó los offsets **0 a 999**. Como no se realizó `commit`, al reiniciar el mismo grupo Kafka no tenía un offset confirmado para esa partición y volvió a entregar los mismos mensajes.

La segunda lectura confirmó:

**Eventos procesados dos veces: 1.000**

Luego se ejecutó `commit()` y quedó confirmado el offset **1000**.

Esto demuestra la semántica **at-least-once**: se evita perder eventos, pero pueden existir duplicados cuando una falla ocurre después de procesar un mensaje y antes de confirmar el offset.

### ¿Nuestro consumidor es idempotente?

**No.**

El consumidor incrementa contadores, suma ratings y aumenta el conteo de reseñas de 1 estrella cada vez que recibe un evento. Si Kafka entrega nuevamente la misma reseña, estas operaciones se ejecutan otra vez y modifican el estado por segunda vez.

Para hacerlo idempotente sería necesario identificar cada evento de manera única y deduplicarlo, guardar los eventos ya procesados o utilizar un motor de streaming con manejo de estado y garantías de procesamiento más robustas.

## 7. Reflexión final

El consumidor Python mantiene su estado solamente en memoria local, por lo que un reinicio o rebalanceo no transfiere automáticamente las cuentas acumuladas.  
Kafka conserva el orden dentro de cada partición, pero no existe un orden global entre las seis particiones del topic.  
La semántica *at-least-once* evita perder información, aunque puede producir eventos duplicados si existe una falla antes del `commit`.  
Para un sistema productivo sería necesario incorporar estado distribuido, deduplicación, tolerancia a fallas, ventanas temporales y recuperación automática.  
Un motor como Spark Structured Streaming permitiría resolver estas necesidades de forma distribuida y será una evolución natural del procesamiento realizado en este práctico.

## Entrega

La carpeta `work/practico4` debe conservar:

```text
practico4_g4_FINAL.ipynb
productor.py
consumidor.py
g4.py
panel.py
```

El topic principal `g4.resenas` no se elimina.